# Task 2: SigLIP + Qwen3-0.6B vision-language model

Everything that has to survive a disconnect (splits, features, checkpoints, results) is written to Google Drive.
Training resumes automatically from the last checkpoint (every 250 updates): re-run the setup cells, then the same training cell.

A T4 is enough for every step.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
ROOT = '/content/drive/MyDrive/atml_pa3/task2'
SPLITS, FEATS_DRIVE, RUNS, RES = f'{ROOT}/splits', f'{ROOT}/features', f'{ROOT}/runs', f'{ROOT}/results'
for d in (SPLITS, RUNS, RES):
    os.makedirs(d, exist_ok=True)
FEATS = '/content/features'  # local copy used for training
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
REPO = 'github.com/umairs5/atml-pa3.git'
if not os.path.exists('/content/pa3'):
    !git clone -q https://{REPO} /content/pa3
else:
    !cd /content/pa3 && git pull -q
%cd /content/pa3
!pip install -q -r requirements.txt

## Data selection

The selected splits are committed in `splits/task2/` and are copied to Drive from there.
If they are missing, this downloads the COCO 2014 and VQAv2 annotation files (about 300 MB) and reruns the selection with seed 6304, which gives the same files.

In [ ]:
if os.path.exists('/content/pa3/splits/task2/selection_info.json'):
    !cp /content/pa3/splits/task2/*.json {SPLITS}/
elif not os.path.exists(f'{SPLITS}/selection_info.json'):
    !mkdir -p /content/raw
    %cd /content/raw
    !wget -q http://images.cocodataset.org/annotations/annotations_trainval2014.zip
    for f in ['Questions_Train', 'Questions_Val', 'Annotations_Train', 'Annotations_Val',
              'Complementary_Pairs_Train', 'Complementary_Pairs_Val']:
        !wget -q https://s3.amazonaws.com/cvmlp/vqa/mscoco/vqa/v2_{f}_mscoco.zip
    !unzip -oq '*.zip'
    %cd /content/pa3
    !python -m task2.prepare_data --raw /content/raw --out {SPLITS}
!cat {SPLITS}/selection_info.json

## Images and SigLIP features

Only the selected images are downloaded (about 9-10k). SigLIP runs once over them, then is removed from the GPU.
The development/test images are also copied to Drive for the qualitative figures.

In [ ]:
if not os.path.exists(f'{FEATS_DRIVE}/feature_info.json'):
    !python -m task2.download_images --images {SPLITS}/images.json --out /content/coco
    !mkdir -p {ROOT}/eval_images && cp -r /content/coco/val2014 {ROOT}/eval_images/
    !python -m task2.cache_features --images /content/coco --ids {SPLITS}/images.json --out {FEATS}
    !mkdir -p {FEATS_DRIVE} && cp {FEATS}/* {FEATS_DRIVE}/
elif not os.path.exists(f'{FEATS}/features.npy'):
    !mkdir -p {FEATS} && cp {FEATS_DRIVE}/* {FEATS}/
!cat {FEATS_DRIVE}/feature_info.json | head -c 600

## Interface checks

Feature shapes, frozen/trainable parameter sets, connector gradients and split isolation, saved to `results/checks.json`.

In [ ]:
!python -m task2.checks --features {FEATS} --splits {SPLITS} --out {RES}/checks.json

## Smoke test

A few updates of each stage (local disk only) to check memory and seconds per update.

In [ ]:
!python -m task2.train --config task2/configs/vlm196.yaml --features {FEATS} --splits {SPLITS} --out_root /content/smoke --max_updates 5
!rm -rf /content/smoke
!python -m task2.train --config task2/configs/vlm196.yaml --features {FEATS} --splits {SPLITS} --out_root /content/smoke --max_updates 5 --start_stage 1
!rm -rf /content/smoke
!python -m task2.train --config task2/configs/vlm196.yaml --features {FEATS} --splits {SPLITS} --out_root /content/smoke --max_updates 5 --start_stage 2
!rm -rf /content/smoke

## Main run (196 visual tokens)

4,500 updates over three stages. Re-run this cell after a disconnect to resume.

In [ ]:
!python -m task2.train --config task2/configs/vlm196.yaml --features {FEATS} --splits {SPLITS} --out_root {RUNS}

## Evaluation

Final model (stage 3) on both test suites under the four image conditions, plus captions for the development images.
The stage-1/2/3 models are also run on the development suites to see what changes between stages.

In [ ]:
RUN = f'{RUNS}/vlm196'
!python -m task2.evaluate --run_dir {RUN} --stage 3 --features {FEATS} --splits {SPLITS} --suites grounding_test vqa_test --captions --out {RES}/vlm196/test_stage3
for st in [1, 2, 3]:
    !python -m task2.evaluate --run_dir {RUN} --stage {st} --features {FEATS} --splits {SPLITS} --suites grounding_dev vqa_dev --out {RES}/vlm196/dev_stage{st}
!cp {RUN}/train_log.jsonl {RUN}/dev_log.jsonl {RUN}/resources.json {RUN}/calibration.json {RUN}/config.json {RES}/vlm196/

## Optional: 49 visual tokens

Same recipe with 2x2 average pooling before the connector (new connector and adapters).

In [ ]:
!python -m task2.train --config task2/configs/vlm49.yaml --features {FEATS} --splits {SPLITS} --out_root {RUNS}
RUN = f'{RUNS}/vlm49'
!python -m task2.evaluate --run_dir {RUN} --stage 3 --features {FEATS} --splits {SPLITS} --suites grounding_test vqa_test --captions --out {RES}/vlm49/test_stage3
!cp {RUN}/train_log.jsonl {RUN}/dev_log.jsonl {RUN}/resources.json {RUN}/calibration.json {RUN}/config.json {RES}/vlm49/

Disconnect when done so the runtime stops using compute units.

In [ ]:
from google.colab import runtime
drive.flush_and_unmount()
runtime.unassign()